In [1]:
import requests
import pandas as pd
import json
import openai
from openai import OpenAI
import time
import re
import os

# check disease name 2 underwriting

In [2]:
df_dise_2_conc = pd.read_csv("./utils/disease_2_conclusion_v2.csv",keep_default_na=False)
df_dise_2_keyword = pd.read_excel("./中再疾病函数对照表.xlsx",keep_default_na=False)
# df_dise_2_keyword_kl = df_dise_2_keyword[df_dise_2_keyword["KL_match_disease"].apply(lambda x:x!="")]
df_dise_2_keyword_kl = df_dise_2_keyword

In [45]:
for idx,row in df_dise_2_conc.iterrows():
    if row["file_name"] == "肺结节.txt":
        df_dise_2_conc.loc[idx,"disease_name"] = "肺结节"

In [47]:
df_dise_2_conc.to_csv("./utils/disease_2_conclusion_v3.csv",index=False)

In [48]:
df_dise_2_conc[df_dise_2_conc["disease_name"]=="甲状腺癌"].head()

,disease_name,file_name,conclusion
975,甲状腺癌,甲状腺癌.txt,"{'诊断': '病理分类： 乳头状癌、滤泡性癌（55岁以下）', '病情现状': '肿瘤≤2..."
976,甲状腺癌,甲状腺癌.txt,"{'诊断': '病理分类： 乳头状癌、滤泡性癌（55岁以下）', '病情现状': '肿瘤2c..."
977,甲状腺癌,甲状腺癌.txt,"{'诊断': '病理分类： 乳头状癌、滤泡性癌（55岁以下）', '病情现状': '肿瘤<4..."
978,甲状腺癌,甲状腺癌.txt,"{'诊断': '病理分类： 乳头状癌、滤泡性癌（55岁以下）', '病情现状': '肿瘤<4..."
979,甲状腺癌,甲状腺癌.txt,"{'诊断': '病理分类： 乳头状癌、滤泡性癌（55岁以上）', '病情现状': '治疗结束..."


In [3]:
df_dise_2_keyword.head()

,def_name,HBZS_def,HBZS_did,disease_n,disease_list,kwords,CI,MI,ADB,Life,ZZ_match_disease,RZ_match_disease,MZ_match_disease,KL_match_disease
0,JiZhenL,,,肌阵挛,['肌阵挛'],,,,,,dianxian,,,
1,JiaZX_MiMXBB,JiaZXMMXBB,135,甲状腺弥漫性病变,['甲状腺弥漫性病变'],"[['甲状腺', '甲状', '甲壮腺', '甲超'], ['弥漫性', '非均质性改变',...",甲状腺恶性肿瘤（含原位癌）及其复发和转移,甲状腺疾病,,甲状腺恶性肿瘤（含原位癌）及其复发和转移,jiazhuangxianzhongliu,liangxingjiazhuangxianjibing,,
2,JiaZX_QieCSH,,,甲状腺切除术后,['甲状腺切除术后'],,,,,,,liangxingjiazhuangxianjibing,,
3,ZhuiTWX_ZHZ,,,椎体外系综合征,['椎体外系综合征'],,,,,,,,,
4,NongDX_NaoB,,,脓毒症相关性脑病,['脓毒症相关性脑病'],,,ruxian,,,,,,


In [4]:
#构建疾病及其相关关键词map, df来自中再疾病函数对照表
disease_n_2_list = {}
for idx,row in df_dise_2_keyword_kl.iterrows():
    disease_n = row["disease_n"].strip()
    disease_list = eval(row["disease_list"])
    # print(type(disease_list))
    try:
        if disease_n  not in disease_n_2_list and disease_n!="":
            disease_n_2_list[disease_n] = []
        disease_n_2_list[disease_n].extend(disease_list)
    except Exception as es:
        print(es)




''
''
''
''
''


In [6]:
def clear_text(text):
    text = text.strip()
    text = text.replace("\n","")
    text = text.replace("*","")
    text = text.replace("\t","")
    text = text.replace("2","II")
    text = text.replace("1","I")
    return text

disease_ns = disease_n_2_list.keys()
disease_names = list(set(df_dise_2_conc["disease_name"].tolist()))


disease_ns = [clear_text(k) for k in disease_ns]
disease_names = [clear_text(k) for k in disease_names]

print("disease_ns",len(disease_ns))
print("disease_names",len(disease_names))

disease_ns 768
disease_names 227


In [23]:
for k in disease_names:
    print(k)

骨髓增生异常综合征
肺动静脉瘘
滴虫性阴道炎
三叉神经痛
腺瘤、息肉
更年期综合征
神经衰弱
疑似错构瘤的肺结节
新生儿卵圆孔未闭
胆管炎
肌营养不良症家族史
头痛
腰椎间盘突出
心肌病家族史
焦虑症
植物神经功能紊乱
甲状腺手术
巨结肠
急性失血性贫血
甲状腺结节
新生儿房/室间隔缺损
甲减
继发于慢性病的贫血
视网膜出血
前列腺炎
视网膜中央动脉血栓症
胆囊息肉
子宫内膜息肉
失眠症
肾动脉狭窄
泌尿系结石（无高血压和肾功能损害）
阻塞性睡眠呼吸暂停综合征
肾囊肿（排除了肾功能损害的）
鼻炎
食管
扁桃体炎
附件
肺纤维化
梅毒
肛裂
颅内神经纤维瘤
肠梗阻
地中海贫血
酒精性肝病
•急性轴索型神经病•急性特发性炎症性多发性神经病•急性炎症性脱髓鞘性多发性神经病 (AIDP)•急性炎症性多发性神经病 (AIPN)•急性运动性轴索型神经病 (AMAN)•急性运动感觉性轴索型神经病 (AMSAN)•慢性炎症性脱髓鞘性神经病 (CIDP)•慢性炎症性多发性神经病 (CIP)•慢性复发性炎症性多发性神经病 (CIRP)•多发性神经根性神经炎•吉兰-巴雷综合征•格林-巴利综合征•GB综合征（Guillain-Barré）
慢性肾盂肾炎
强直性脊柱炎
心肌炎
抑郁症
病理分类： 乳头状癌、滤泡性癌（55岁以下）
宫颈纳囊
盆腔积液
颅咽管瘤
鼻中隔偏曲
病理分类： 乳头状癌、滤泡性癌（55岁以上）
II型糖尿病
流行性乙型脑炎
肝囊肿
流行性腮腺炎
心包炎/心包积液
皮肌炎
其他心脏结构异常
自发性气胸
色素性视网膜炎
纵膈气肿
脾大
先天性肾畸形/单肾
视神经炎
肝硬化
肾下垂
结直肠癌
阴道壁膨出
新生儿缺氧缺血性脑病
肺水肿
胆囊炎
面神经炎（面瘫）
视网膜血管变性
病理分类：间变性癌、未分化癌、髓样癌等
哮喘
耳聋
股骨头坏死
气管炎/支气管炎
风湿性关节炎
鼻部假体植入
类风湿性关节炎
干燥综合征
消化性溃疡
肠吸收不良
手术
胃息肉
雷诺综合征
中性粒细胞减少症
甲型肝炎
扩张性心肌病
HPV感染
中耳炎
胆结石
腰椎滑脱
子宫内膜增厚（内膜厚度超过I0mm）
EB病毒感染
胸膜炎
肝内胆管结石
缺铁性贫血
肌营养不良症
肝血管瘤
系统性红斑狼疮
视网膜脱离
二尖瓣/三尖瓣
乳腺结节、囊肿、占位、异常回声
I型糖尿病
脑膜炎/脑脊髓膜炎
宫颈上皮内瘤变
丙型肝炎

In [7]:
intersection = set(disease_ns) & set(disease_names)
print(len(intersection))
disease_names_only = set(disease_names) - set(disease_ns)
print(f"disease_name only have:{len(disease_names_only)}")
disease_ns_only = set(disease_ns) - set(disease_names)
print("disease_ns only have:",len(disease_ns_only))

116
disease_name only have:111
disease_ns only have: 652


In [21]:
dict_map_diff_k = {}
for k_i in disease_ns_only:
    for k_j in disease_names_only:
        if (k_i in k_j) or (k_j in k_i):
            dict_map_diff_k[k_i] = k_j

In [22]:
print(len(dict_map_diff_k))
for k,v in dict_map_diff_k.items():
    print(k,":",v)


67
心包炎 : 心包炎/心包积液
紫癜 : 过敏性紫癜
子宫萎缩 : 子宫
疱疹 : 带状疱疹
子宫切除手术 : 子宫
胃-食管返流性疾病 : 食管
心肌病 : 肥厚性心肌病、限制性心肌病、非致密化心肌病、致心律失常性心肌病等
肾盂肾炎 : 急性细菌性肾盂肾炎、急性肾盂肾炎
突发耳聋 : 耳聋
子宫内膜腺癌 : 子宫
溶血病 : 新生儿溶血病
肺大疱 : 肺大疱（肺囊肿）
慢性胰腺炎 : 胰腺炎
腮腺炎 : 流行性腮腺炎
食管良性肿瘤 : 食管
支气管扩张症 : 支气管扩张
先天性肾畸形 : 先天性肾畸形/单肾
慢性丙型肝炎 : 丙型肝炎
卵巢切除手术 : 手术
肝炎 : 丙型肝炎
肿瘤 : 纵膈肿瘤
肺结节病 : 肺结节
肾炎 : 急性肾病综合征（急性肾小球肾炎、感染后肾小球肾炎）
儿童乳腺发育 : 儿童乳腺
巴氏食管 : 食管
卵圆孔未闭 : 新生儿卵圆孔未闭
子宫内膜增生 : 子宫
急性丙型肝炎 : 丙型肝炎
子宫腺肌病 : 子宫
气胸 : 自发性气胸
高血压 : 泌尿系结石（无高血压和肾功能损害）
溃疡性结肠炎 : 慢性结/直肠炎/溃疡性结肠炎
食管癌 : 食管
子宫肥大 : 子宫
错构瘤 : 肺错构瘤
肾囊肿 : 肾囊肿（排除了肾功能损害的）
心肌损害 : 新生儿心肌损害
粟粒性结核病 : 结核病
多囊肾 : 多囊肾/家族史
肺炎 : 肺炎（不包括新冠肺炎）
颅脑损伤 : 脑损伤
乳腺炎 : 乳腺炎、脓肿
急性胰腺炎 : 胰腺炎
子宫肉瘤 : 子宫
喉炎 : 咽喉炎
肠炎 : 慢性结/直肠炎/溃疡性结肠炎
癌 : 宫颈癌
肾病综合征 : 肾病综合征（慢性）
脑膜炎 : 脑膜炎/脑脊髓膜炎
肺囊肿 : 肺大疱（肺囊肿）
气管炎 : 气管炎/支气管炎
子宫内膜非典型增生 : 子宫
营养不良 : 肌营养不良症家族史
胆管结石 : 肝内胆管结石
心包积液 : 心包炎/心包积液
损伤 : 脑损伤
贫血 : 巨幼细胞性贫血
食管炎 : 食管
子宫内膜癌 : 子宫
乳腺结节 : 乳腺结节、囊肿、占位、异常回声
子宫炎症 : 子宫
静脉曲张 : （下肢）静脉曲张
阴道炎 : 淋菌性阴道炎
硬化性胆管炎 : 胆管炎
结肠炎 : 慢性结/直肠炎/溃疡性结肠炎
急性肾小球肾炎 : 急性肾病综合征（急性肾小球肾炎、感染后肾小球肾炎）
囊肿 : 肺大疱（肺囊肿）


In [11]:
# print("disease_ns_only")
# for k in disease_ns_only:
#     if k not in dict_map_diff_k.keys():
#         print(k)

In [14]:
df_test = pd.read_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch.csv",keep_default_na=False)

In [19]:
print(set(df_test["disease_name"].tolist()))
for k_n in set(df_test["disease_name"].tolist()):
    if k_n not in intersection:
        print(k_n)

{'子宫内膜息肉', '', '乳腺结节、囊肿、占位、异常回声', '脂肪肝*', '泌尿系结石（无高血压和肾功能损害）', '胆结石', '甲状腺结节', '肺结节', '肝血管瘤', '肝囊肿', '子宫肌瘤', '胆囊息肉'}

乳腺结节、囊肿、占位、异常回声
脂肪肝*
泌尿系结石（无高血压和肾功能损害）
胆结石
肺结节


In [ ]:
"""
胆囊疾病:胆结石
肺结节病:肺结节
脂肪肝:脂肪肝*
乳腺结节:乳腺结节、囊肿、占位、异常回声
肾结石:泌尿系结石（无高血压和肾功能损害）
"""

In [18]:
print(set(df_test["disease_name"].tolist()))
for k_n in set(df_test["disease_name"].tolist()):
    if k_n in intersection:
        print(k_n)


{'子宫内膜息肉', '', '乳腺结节、囊肿、占位、异常回声', '脂肪肝*', '泌尿系结石（无高血压和肾功能损害）', '胆结石', '甲状腺结节', '肺结节', '肝血管瘤', '肝囊肿', '子宫肌瘤', '胆囊息肉'}
子宫内膜息肉
甲状腺结节
肝血管瘤
肝囊肿
子宫肌瘤
胆囊息肉


In [17]:
for idx,row in df_test.iterrows():
    disease_name = row["disease_name"]
    if disease_name in intersection:
        print(disease_name)

子宫内膜息肉
子宫肌瘤
甲状腺结节
胆囊息肉
甲状腺结节
肝血管瘤
肝囊肿
甲状腺结节
胆囊息肉
甲状腺结节
甲状腺结节
甲状腺结节


In [37]:
#大模型
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:50020/v1"
)


def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-14B",
        messages=messages,
        logprobs=False,
        # stream=True  # 开启流式输出
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

# 编辑距离计算疾病相似度

In [5]:
def edit_distance(str1, str2):
    """计算两个字符串的编辑距离

    Args:
        str1: 字符串1
        str2: 字符串2

    Returns:
        int: 编辑距离
    """

    m = len(str1)
    n = len(str2)

    # 初始化二维数组dp，dp[i][j]表示str1[:i]和str2[:j]的编辑距离
    dp = [[i+j for j in range(n+1)] for i in range(m+1)]
    for i in range(1, m+1):
        dp[i][0] = i
    for j in range(1, n+1):
        dp[0][j] = j

    for i in range(1, m+1):
        for j in range(1, n+1):
            if str1[i-1] == str2[j-1]:
                cost = 0
            else:
                cost = 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)

    return dp[m][n]

def normalized_similarity(str1, str2):
    distance = edit_distance(str1, str2)
    max_len = max(len(str1), len(str2))
    similarity = 1 - distance / max_len
    return similarity

# 示例用法
str1 = "kitten"
# str2 = "sitting"
str2 = "kitte"

distance = edit_distance(str2, str1)
print("编辑距离:", distance)

distance_nor = normalized_similarity(str1,str2)
print(round(distance_nor,4))


编辑距离: 1
0.8333


In [6]:
# 计算疾病相似度，并返回top_n最相近的疾病
def disease_similarity(input_dise,disease_n_2_list,top_n=3):
    disease_similarity_score = {}
    for k_dise,v_diseKey in disease_n_2_list.items():
        disease_similarity_score[k_dise] = max([normalized_similarity(input_dise,keyw) for keyw in v_diseKey])

    sorted_dict = sorted(disease_similarity_score.items(), key=lambda x: x[1], reverse=True)
    result =sorted_dict[:top_n]
    return result
    

# 读取 中再疾病函数对照表的disease_n 到 disease_2_concluson中的disease_name的 map
# 读取 中再疾病函数对照表的disease_n 到 key word 的map

In [7]:
with open("./utils/KL_disease_n_2_name_map.json","r",encoding="utf-8") as f:
    map_key_2_dise = json.load(f)


with open("./utils/KL_disease_n_2_keyword.json","r",encoding="utf-8")as f:
    disease_n_2_list = json.load(f)

In [55]:
#测试样本
df = pd.read_csv("./result_recall_disease/核保结论_规则引擎_vs_RAG_disease_recall.csv",keep_default_na=False)

#昆仑核保结论知识
df_dise_2_conc = pd.read_csv("./utils/disease_2_conclusion_v4.csv",keep_default_na=False)  

In [60]:
df_dise_2_conc[df_dise_2_conc["disease_name"]=="肝内胆管结石"]["disease_name"].tolist()

['肝内胆管结石', '肝内胆管结石']

In [56]:
df_dise_2_conc[df_dise_2_conc["file_name"]=="妇科肿物.txt"].head()

,disease_name,file_name,conclusion
1031,子宫/内膜肿瘤,妇科肿物.txt,"{""疾病"":""子宫\/内膜肿瘤"",""资料"":""病历、妇科超声、病理结果"",""检查结果"":""现..."
1032,子宫/内膜肿瘤,妇科肿物.txt,"{""疾病"":""子宫\/内膜肿瘤"",""资料"":""病历、妇科超声、病理结果"",""检查结果"":""术..."
1033,子宫/子宫肉瘤,妇科肿物.txt,"{""疾病"":""子宫\/子宫肉瘤"",""资料"":""病历、妇科超声、病理结果"",""检查结果"":"",..."
1034,子宫/滋养细胞肿瘤\n（葡萄胎）,妇科肿物.txt,"{""疾病"":""子宫\/滋养细胞肿瘤\n（葡萄胎）"",""资料"":""病历、妇科超声、HCG"",""..."
1035,附件/多囊卵巢综合征,妇科肿物.txt,"{""疾病"":""附件\/多囊卵巢综合征"",""资料"":""妇科超声、血脂、OGTT or 血糖+糖..."


In [10]:
df.columns

Index(['姓名', 'disease_name', '编号（身份证号）', '性别\n（1:男，2:女，0:未知）', '医院', '日期',
       '临床诊断（化验项、疾病等以下划线拼接）', '账单金额（数字）', '年龄', '编号（案件编号）', '图片名/文件名',
       'attach id', '图片分类/票据类别', '影像报告内容image_report',
       'data_source：类别 如昆仑体检告知', '线上页面与结果', 'RAG核保结论', 'recall_query',
       'recall_disease', 'recall_conclsion'],
      dtype='object')

In [61]:
map_key_2_dise = {
    "胆囊疾病":"胆结石",
    "肺结节病":"肺结节",
    "脂肪肝":"脂肪肝*",
    "乳腺结节":"乳腺结节、囊肿、占位、异常回声",
    "肾结石":"泌尿系结石（无高血压和肾功能损害）",
    "心率不齐":"心率失常",
    "肺动脉瓣关闭不全":"肺动脉瓣疾病",
    "胆管结石":"肝内胆管结石",
    "卵巢囊肿":"附件/多囊卵巢综合征"
}

In [30]:
all_dise_conc_keys = set(df_dise_2_conc["disease_name"].tolist())

In [51]:
gender_map = {2:"女性",1:"男"}
results_underwriting = []
recall_query = []
recall_keys = []
recall_time_use = []
for idx,row in df.iterrows():
    
    gender = gender_map.get(row["性别\n（1:男，2:女，0:未知）"],"未知")
    age = row["年龄"]
    diagnose = row["临床诊断（化验项、疾病等以下划线拼接）"]
    image_report = row["影像报告内容image_report"].replace('\\n',"")
    image_report = image_report.replace("\\",'').strip("\"").replace("\'","\"")
    basic_info = f"年龄:{age},性别:{gender},临床诊断:{diagnose},影像报告:{image_report}"
    # print("input:",diagnose)
    # print("disease name:",row["disease_name"])
    print("病人基本信息",basic_info)
    # print(image_report)
    # if image_report != "":
    #     image_report = json.loads(image_report)
    #     print(type(image_report))
    #     query = diagnose + image_report["des_dic"] + image_report["con_dic"]
    # else:
    #     query = diagnose    
    
    #召回方案：根据临床诊断召回相应核保结论
    query = diagnose
    
    
    try:
        s_time = time.time()
        #获得相似疾病name
        map_disease_name= disease_similarity(query,disease_n_2_list)
        disease_key = map_disease_name[0][0]
        if disease_key in all_dise_conc_keys:
            max_simi_key = disease_key
        else:
            max_simi_key = map_key_2_dise.get(disease_key,"")
        # max_simi_dise_names = map_key_2_dise.get(map_disease_name[0][0],[map_disease_name[0]])
        #获得相似疾病的结论
        recall_kbs = df_dise_2_conc[df_dise_2_conc["disease_name"]==max_simi_key]["conclusion"].tolist() 
        time_use = time.time() - s_time
        print("recall time use:",time_use)
        recall_time_use.append(time_use)
        ans_list = recall_kbs

    except Exception as es:
        print(es)
        ans_list = []

    #构建prompt
    output_struct = {"重疾险": "延期", "防癌": "延期", "护理": "肾功能异常延期", "医疗险": "延期", "意外险": "肾功能异常延期"}
    prompt = """
            你是一个保险公司的专业核保老师，根据提供的病人基本信息和给定的核保结论返回最接近的核保结论。
            病人基本信息:{basic_info}
            核保结论:{ans_list}
            注意：
                1、核保结论的返回格式为:{output_struct}
                2、不要有其他信息说明
                3、诺核保结论为空则不返回最终核保结论
            """
    input_prompt = prompt.format(basic_info=basic_info,ans_list=ans_list,output_struct=output_struct)
    # print("input_prompt:",input_prompt)


    #大模型结论生成
    input = [{"role": "user", "content": input_prompt}]
    result = qa_base(input)

    results_underwriting.append(result)
    recall_query.append(ans_list)
    recall_keys.append(disease_key)

    

病人基本信息 年龄:34,性别:女性,临床诊断:宫腔内稍高回声团,考虑子宫内膜息肉,影像报告:{"report_name": "", "des_dic": "经阴道三维超声检查：后位子宫  切面形态规则  体积不大  包膜完整  肌层回声分布均匀  未见明显肿块声像  内膜厚约8mm宫腔内可见一个大小约9x4mm稍高回声团，  边界清  CDFI:  其内未见明显血流信号。三维成像  ：双侧宫角可见显示  官腔形态未见异常。双侧附件区未见明显异常包块回声。子宫直肠窝未见游离液性暗区。", "con_dic": "宫腔内稍高回声团,  考虑子宫内膜息肉。双侧附件区未见明显异常包块回声。"}
recall time use: 0.6081547737121582
病人基本信息 年龄:29,性别:女性,临床诊断:子宫肌瘤可能,影像报告:{"report_name": "", "des_dic": "【子宫】  【经阴道】子宫位置：  前位：  子宫大小：  长径  60mm,  左右径  65mm,  前后径  59mm;子宫形态：  不规则：  子宫回声：  不均匀：  肌层彩色血流星点状,内膜厚度12mm    宫颈长度  38mm子宫后壁突起中低回声区：  54*50*50mm", "con_dic": "子宫肌瘤可能。"}
recall time use: 0.22733402252197266
病人基本信息 年龄:35,性别:男,临床诊断:双肾结石(沙粒样),影像报告:{"report_name": "", "des_dic": "双肾轮廓清晰。  切面形态大小正常：  表面光滑：  皮质回声正常  皮髓质分界清晰。  于双肾盏可见点状强回声  后伴浅声影。  双肾集合系统排列规则，  未见分离。  CDFI  未见明显异常血流信号。", "con_dic": "双肾结石（沙粒样）"}
recall time use: 0.3364131450653076
病人基本信息 年龄:35,性别:男,临床诊断:脂肪肝（中度）,影像报告:{"report_name": "", "des_dic": "肝脏切面轮廊清晰，  右叶斜径约171mm  肝内回声前场增强细密，  后场稀疏衰减：  分布不均匀  肝内管道结构隐约可见  出肝光带模糊。  肝内胆管未见明显扩张  门静

In [52]:
sum(recall_time_use)/len(recall_time_use)

0.32474566996097565

In [40]:
df.columns

Index(['姓名', 'disease_name', '编号（身份证号）', '性别\n（1:男，2:女，0:未知）', '医院', '日期',
       '临床诊断（化验项、疾病等以下划线拼接）', '账单金额（数字）', '年龄', '编号（案件编号）', '图片名/文件名',
       'attach id', '图片分类/票据类别', '影像报告内容image_report',
       'data_source：类别 如昆仑体检告知', '线上页面与结果', 'RAG核保结论', 'recall_query',
       'recall_disease', 'recall_conclsion'],
      dtype='object')

In [53]:
df_new = df.drop(['RAG核保结论', 'recall_query',
       'recall_disease', 'recall_conclsion'],axis=1)

In [54]:
df_new["RAG核保结论"] = results_underwriting
df_new["recall_disease"] = recall_keys
df_new["recall_conclusions"] = recall_query
df_new.to_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch_20241203.csv",index=False)

In [37]:
len_get_recall = len([rec_res for rec_res in recall_query if len(rec_res)!=0])
print(len_get_recall)
print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")

32
召回率：100.0%


In [38]:
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_FullTextSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_SemanticSearch.csv")
# df_full = pd.read_csv("./核保结论_规则引擎_vs_RAG_HybridSearch.csv")
# df_full = pd.read_csv("./results_edit_distance/核保结论_规则引擎_vs_RAG_editDistanceSearch.csv")


# recall_query = df_full["recall_query"].tolist()
# len_get_recall = len([rec_res for rec_res in recall_query if len(eval(rec_res))!=0])
# print(len_get_recall)
# print(f"召回率：{round(len_get_recall/len(recall_query)*100,2)}%")
bad_dise = []
for idx,row in df.iterrows():
    if len(row["recall_query"]) == 0:
        bad_dise.append(row["临床诊断（化验项、疾病等以下划线拼接）"])
print(len(bad_dise))
print(bad_dise)

0
[]


In [25]:
df_dise_2_conc[df_dise_2_conc["file_name"]=="子宫.txt"].head(30)

,disease_name,file_name,conclusion
812,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
813,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
814,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '功能失调性子宫出血', '资料': '病历、妇科超声..."
815,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '非功能失调性子宫出血', '资料': '病历、妇科超..."
816,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '绝经后子宫出血', '资料': '病历、妇科超声、血..."
817,子宫出血,子宫.txt,"{'疾病': '子宫出血', '': '待进行子宫切除', '资料': '病历、妇科超声、血..."
818,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
819,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
820,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
821,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."


In [33]:
query_dise = bad_dise[1]
print(query_dise)
results_query = disease_similarity(query_dise,disease_n_2_list)
print(results_query)
print(results_query[0][0])
map_disease_name = map_key_2_dise.get(results_query[0][0],[results_query[0]])
print(map_disease_name)
df_dise_2_conc[df_dise_2_conc["disease_name"]==map_disease_name].head()

子宫肌瘤可能
[('子宫肌瘤', 0.6666666666666667), ('子宫肉瘤', 0.5), ('子宫切除手术', 0.33333333333333337)]
子宫肌瘤
子宫肌瘤


,disease_name,file_name,conclusion
818,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
819,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
820,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."
821,子宫肌瘤,子宫.txt,"{'疾病': '子宫肌瘤', '': '子宫肌瘤、子宫腺肌瘤、子宫纤维瘤', '资料': '..."


In [62]:
#评测
"""
四个召回错误
2个召回正确，生成结论错误
"""

'\n四个召回错误\n2个召回正确，生成结论错误\n'